# 01 - Hugging Face models from .NET

Loading a real checkpoint, tokenizing exactly the way it was trained, and running it.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

## Setup

Set `HF_TOKEN` in your environment for private or gated repositories, and for a much higher
anonymous rate limit.

In [ ]:
// HF.Net is on nuget.org, so these restore straight into the notebook.
#r "nuget: Gravicode.HFNet.GraviHub, 0.2.0"
#r "nuget: Gravicode.HFNet.GraviTokenizers, 0.2.0"
#r "nuget: Gravicode.HFNet.GraviTransformers, 0.2.0"
#r "nuget: Gravicode.HFNet.GraviDatasets, 0.2.0"

using System;
using System.Linq;

## 1. What is on the Hub?

File listings come from the tree API, so every entry carries its real size - which is what
lets you choose between two weight files rather than guessing.

In [ ]:
using Gravicode.HFNet.GraviHub;

var info = Hub.ModelInfo("prajjwal1/bert-tiny");
Console.WriteLine(info);
Console.WriteLine($"safetensors: {info.HasSafeTensors}");

foreach (var file in info.Files) Console.WriteLine($"  {file}");

This one publishes only `pytorch_model.bin`, which a loader that understands safetensors
alone cannot open. HF.Net reads both.

## 2. Tokenize

Offsets index the **original, untouched** string, so a span can always be reported as a
substring rather than as token indices nobody can interpret.

In [ ]:
using Gravicode.HFNet.GraviTokenizers;

var tokenizer = HfTokenizer.FromPretrained("bert-base-uncased");

const string Text = "Hello, world! Tokenizers are unbelievable.";
var encoding = tokenizer.Encode(Text);

Console.WriteLine(string.Join(' ', encoding.Tokens));
Console.WriteLine(string.Join(' ', encoding.Ids));

Those are the same ids the Python implementation produces. The offsets recover the original
casing, because normalization runs per pre-token rather than over the whole string:

In [ ]:
for (var i = 0; i < encoding.Length; i++)
    Console.WriteLine($"{encoding.Tokens[i],-14} {encoding.Offsets[i],-10} '{encoding.Span(Text, i)}'");

## 3. Load a model and ask it something

Fill-mask is the sharpest check that a checkpoint loaded correctly. A model with a
transposed weight or a shifted position embedding still produces plausible-looking vectors -
but it does not answer this with *paris*.

In [ ]:
using Gravicode.HFNet.GraviTransformers;

// About 440 MB on the first run; one request after that.
using var model = TransformerModel.Load("bert-base-uncased");

Console.WriteLine(model);
Console.WriteLine(model.Report);

In [ ]:
foreach (var fill in model.FillMask("The capital of France is [MASK].", topK: 5))
    Console.WriteLine($"{fill.Token,-14} {fill.Score:P2}");

## 4. Embeddings and similarity

`Embed` mean-pools over the tokens rather than taking `[CLS]`: on a model that has not been
fine-tuned for sentence similarity, `[CLS]` is close to constant and makes every pair of
sentences look alike.

In [ ]:
Console.WriteLine(model.Similarity("the cat sat on the mat",
                                   "the dog sat on the rug"));
Console.WriteLine(model.Similarity("the cat sat on the mat",
                                   "quarterly earnings beat expectations"));

## 5. Classification

A fine-tuned checkpoint carries its own head and its own label names, so nothing here has to
be told what the classes are.

In [ ]:
using var sst = TransformerModel.Load("distilbert-base-uncased-finetuned-sst-2-english");

Console.WriteLine(string.Join(", ", sst.Labels));

foreach (var text in new[] { "I absolutely loved this film.", "A complete waste of time." })
    Console.WriteLine($"{text,-34} -> {sst.Predict(text, topK: 1)[0]}");

## Where to go next

- `TransformerModel.Load` **refuses** decoder-only models (GPT, Llama) rather than
  half-loading them - filling an encoder from a decoder's weights produces a model that runs
  and returns nonsense.
- For throughput, export to ONNX and use `GraviOptimum` - see notebook 02.
- Documentation: [docs/](../docs/), and in Bahasa Indonesia at [docs/id/](../docs/id/).